# [M3S03] - Ex. 4 — Reprodutibilidade com random_state

**Objetivo:** garantir que os resultados do modelo sejam idênticos a cada execução, definindo `random_state` na divisão dos dados e nos modelos.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
df = pd.read_csv('../../data/input/csv/dados_AP - [M3S01] - Ex. 1 até [M3S03] - Ex. 4.csv')
X = df[['investimento_marketing', 'preco_medicamento', 'mes']]
y = df['demanda']

## 1. Demonstração: sem vs com random_state

Executar a célula abaixo múltiplas vezes **sem** `random_state` produz R² diferente a cada vez.

In [3]:
# Sem random_state — resultado muda a cada execução
resultados_sem = []
for _ in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    r2 = r2_score(y_test, LinearRegression().fit(X_train, y_train).predict(X_test))
    resultados_sem.append(round(r2, 6))

print('R² sem random_state (5 runs):', resultados_sem)
print(f'Variação: {max(resultados_sem) - min(resultados_sem):.6f}')

R² sem random_state (5 runs): [0.956715, 0.956822, 0.958252, 0.956502, 0.952711]
Variação: 0.005541


In [4]:
# Com random_state — resultado idêntico a cada execução
resultados_com = []
for _ in range(5):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    r2 = r2_score(y_test, LinearRegression().fit(X_train, y_train).predict(X_test))
    resultados_com.append(round(r2, 6))

print('R² com random_state=42 (5 runs):', resultados_com)
print(f'Variação: {max(resultados_com) - min(resultados_com):.6f}')

R² com random_state=42 (5 runs): [0.954393, 0.954393, 0.954393, 0.954393, 0.954393]
Variação: 0.000000


## 2. Pipeline com random_state definido em todos os pontos

In [5]:
SEED = 42  # ponto único de controle — alterar aqui muda tudo de forma consistente

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

modelo_lr   = LinearRegression()                          # sem parâmetro aleatório
modelo_tree = DecisionTreeRegressor(random_state=SEED)    # com random_state

modelo_lr.fit(X_train, y_train)
modelo_tree.fit(X_train, y_train)

r2_lr   = r2_score(y_test, modelo_lr.predict(X_test))
r2_tree = r2_score(y_test, modelo_tree.predict(X_test))

print(f'R² Regressão Linear  : {r2_lr:.4f}')
print(f'R² Árvore de Decisão : {r2_tree:.4f}')

R² Regressão Linear  : 0.9544
R² Árvore de Decisão : 0.9874


## 3. Por que a reprodutibilidade é importante?

### Por que é importante em projetos reais?

Em projetos de dados, múltiplas pessoas trabalham no mesmo código em momentos diferentes — revisores, colegas de equipe, o próprio autor semanas depois. Sem reprodutibilidade:

- Uma métrica reportada (ex.: R² = 0.9544) não pode ser **verificada** por outra pessoa
- O modelo em produção pode ter comportamento diferente do avaliado durante o desenvolvimento
- Comparações entre modelos ficam inválidas se cada execução usa uma divisão de dados diferente

### O que pode acontecer sem esse controle?

- **Resultados inconsistentes:** a mesma métrica varia entre execuções, tornando difícil saber se uma melhoria é real ou sorte da divisão
- **Depuração impossível:** um bug encontrado em uma execução pode não aparecer na próxima, pois os dados de treino/teste mudaram
- **Decisões de negócio baseadas em sorte:** um modelo pode parecer ótimo em uma divisão favorável e fraco em outra, levando a escolhas erradas